# LeetCode 4056 — Number of Intersecting Interval Pairs I bằng Python

Link đề: https://leetcode.com/problems/number-of-intersecting-interval-pairs-i/ · Độ khó: **Easy** · Chủ đề: Array, Sorting, Two Pointers

Skeleton LeetCode:

```python
class Solution:
    def countIntersectingIntervals(self, intervals: List[List[int]]) -> int:
        ...
```

Ghi chú: đề và các ví dụ lấy từ đề gốc (Weekly Contest 520, câu 1); mọi đáp án trong notebook đều được kiểm chứng bằng code, kể cả bằng một lời giải độc lập so sánh tập điểm nguyên của hai đoạn.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho `n` đoạn **đóng** `[start, end]`; đếm số **cặp chỉ số** `(i, j)` với `i < j` mà hai đoạn **có ít nhất một điểm chung** (kể cả khi chỉ chạm nhau ở đầu mút).
2. Biết **điều kiện giao nhau** của hai đoạn đóng: `max(l1, l2) <= min(r1, r2)`, hay dạng phủ định: **rời nhau** ⇔ `r1 < l2` hoặc `r2 < l1`.
3. Đi từ **duyệt mọi cặp `O(n²)`** → **đếm phần bù** (`tổng số cặp − số cặp rời nhau`) bằng `bisect` → **hai con trỏ trên hai mảng sắp xếp riêng** → **mảng đếm / tổng tiền tố** → **quét đường thẳng (sweep line)**.
4. Hiểu vì sao **đếm cặp rời nhau** dễ hơn đếm cặp giao nhau: mỗi cặp rời nhau có một đoạn "kết thúc trước" duy nhất, nên đếm đúng một lần.
5. Tránh các lỗi kinh điển: dùng `<` thay vì `<=` (bỏ qua trường hợp chạm đầu mút), đếm cặp có thứ tự (gấp đôi), dùng `<=` sai chỗ ở con trỏ, quên sắp `ends` riêng biệt, và chỉ kiểm tra một chiều của quan hệ giao nhau.

## 2. Đề bài & ví dụ

Cho mảng hai chiều `intervals` gồm `n` phần tử, trong đó `intervals[i] = [start_i, end_i]` là **đoạn đóng** từ `start_i` đến `end_i`.

Trả về **số cặp chỉ số** `(i, j)` thỏa `0 <= i < j < n` mà `intervals[i]` và `intervals[j]` **giao nhau**.

Hai đoạn giao nhau nếu chúng có **ít nhất một điểm chung**, kể cả khi chỉ chung một đầu mút.

**Ví dụ 1**

- Input: `intervals = [[1,2],[2,3],[3,4]]`
- Output: `2`
- Giải thích: `[1,2]` và `[2,3]` giao nhau tại điểm `2`; `[2,3]` và `[3,4]` giao nhau tại điểm `3`; `[1,2]` và `[3,4]` không giao nhau.

**Ví dụ 2**

- Input: `intervals = [[1,5],[2,4],[3,6]]`
- Output: `3`
- Giải thích: `[1,5]` và `[2,4]` giao nhau tại `[2,4]`; `[1,5]` và `[3,6]` tại `[3,5]`; `[2,4]` và `[3,6]` tại `[3,4]`.

**Ví dụ 3**

- Input: `intervals = [[1,2],[3,4],[5,6]]`
- Output: `0`
- Giải thích: không có cặp nào giao nhau.

**Ràng buộc:** `2 <= n == intervals.length <= 100`; `intervals[i] = [start_i, end_i]`; `0 <= start_i <= end_i <= 100`.

In [1]:
from typing import List
import random
from bisect import bisect_left
from itertools import accumulate, combinations

examples = [
    ([[1, 2], [2, 3], [3, 4]], 2),
    ([[1, 5], [2, 4], [3, 6]], 3),
    ([[1, 2], [3, 4], [5, 6]], 0),
]
for intervals, expected in examples:
    print(f"intervals={intervals}, kỳ vọng {expected}")

intervals=[[1, 2], [2, 3], [3, 4]], kỳ vọng 2
intervals=[[1, 5], [2, 4], [3, 6]], kỳ vọng 3
intervals=[[1, 2], [3, 4], [5, 6]], kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Đoạn đóng, giao nhau và "rời nhau"

Đoạn đóng `[l, r]` gồm mọi điểm `x` với `l <= x <= r` (có cả hai đầu mút). Hai đoạn **rời nhau** khi đoạn này kết thúc **trước hẳn** khi đoạn kia bắt đầu: `r1 < l2` hoặc `r2 < l1`. Dấu **`<` chặt** là điểm mấu chốt: `[1,2]` và `[2,3]` có `r1 = 2 = l2`, không thỏa `r1 < l2` nên **không** rời nhau, tức giao nhau tại điểm `2`.

In [2]:
def disjoint(a, b):
    """Hai đoạn đóng a = [l1, r1], b = [l2, r2] rời nhau?"""
    return a[1] < b[0] or b[1] < a[0]


pairs = [([1, 2], [2, 3]), ([1, 2], [3, 4]), ([1, 5], [2, 4]), ([3, 4], [1, 5]), ([2, 2], [2, 2])]
for a, b in pairs:
    print(a, b, "rời nhau" if disjoint(a, b) else "giao nhau")

[1, 2] [2, 3] giao nhau
[1, 2] [3, 4] rời nhau
[1, 5] [2, 4] giao nhau
[3, 4] [1, 5] giao nhau
[2, 2] [2, 2] giao nhau


### 3.2. Điều kiện giao nhau gọn: `max(l1, l2) <= min(r1, r2)`

Phần giao của hai đoạn (nếu có) bắt đầu ở `max(l1, l2)` và kết thúc ở `min(r1, r2)`. Nó khác rỗng khi `max(l1, l2) <= min(r1, r2)`. Điều này tương đương "không rời nhau": `l1 <= r2 và l2 <= r1`.

In [3]:
def intersect(a, b):
    return max(a[0], b[0]) <= min(a[1], b[1])


for a, b in pairs:
    assert intersect(a, b) == (not disjoint(a, b))
    lo, hi = max(a[0], b[0]), min(a[1], b[1])
    print(a, b, "phần giao:", [lo, hi] if lo <= hi else "rỗng")

[1, 2] [2, 3] phần giao: [2, 2]
[1, 2] [3, 4] phần giao: rỗng
[1, 5] [2, 4] phần giao: [2, 4]
[3, 4] [1, 5] phần giao: [3, 4]
[2, 2] [2, 2] phần giao: [2, 2]


### 3.3. Đếm số cặp: `n(n−1)/2` và `combinations`

Số cách chọn 2 chỉ số khác nhau `i < j` trong `n` phần tử là `n(n−1)/2` (tổ hợp `C(n, 2)`). Dùng `//` để chia nguyên. `itertools.combinations(range(n), 2)` sinh tất cả cặp `(i, j)` với `i < j`; ta dùng nó để kiểm chứng.

In [4]:
n = 4
print(n * (n - 1) // 2)                    # 6
print(list(combinations(range(n), 2)))     # [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
for n in range(2, 7):
    assert n * (n - 1) // 2 == len(list(combinations(range(n), 2)))

6
[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]


### 3.4. Rút các trường `start` và `end` rồi `sorted`

Từ `intervals = [[1,5],[2,4],[3,6]]` ta lấy `starts = [s for s, e in intervals]` và `ends = [e for s, e in intervals]`. `sorted(list)` trả list mới đã sắp tăng. Chú ý: ta sắp **riêng** `starts` và `ends`, nên chúng không còn ghép đôi với nhau — đó là chủ ý ở Cách 3.

In [5]:
intervals = [[1, 5], [2, 4], [3, 6]]
starts = sorted(s for s, e in intervals)
ends = sorted(e for s, e in intervals)
print(starts, ends)       # [1, 2, 3] [4, 5, 6]

[1, 2, 3] [4, 5, 6]


### 3.5. `bisect_left`: đếm số phần tử nhỏ hơn `x` trong list đã sắp

`bisect_left(a, x)` trả về vị trí chèn `x` vào list sắp tăng `a` sao cho nó đứng **trước** các phần tử bằng `x`. Vị trí đó đúng bằng **số phần tử `< x`** trong `a`. Chạy trong `O(log n)`.

In [6]:
a = [2, 3, 3, 5, 8]
for x in [1, 2, 3, 4, 9]:
    print(f"bisect_left(a, {x}) = {bisect_left(a, x)}  | số phần tử < {x}: {sum(v < x for v in a)}")

bisect_left(a, 1) = 0  | số phần tử < 1: 0
bisect_left(a, 2) = 0  | số phần tử < 2: 0
bisect_left(a, 3) = 1  | số phần tử < 3: 1
bisect_left(a, 4) = 3  | số phần tử < 4: 3
bisect_left(a, 9) = 5  | số phần tử < 9: 5


### 3.6. Mảng đếm và tổng tiền tố với `accumulate`

Khi các giá trị nhỏ (ở đây `0..100`), ta có thể đếm bằng mảng thay vì sắp xếp: `cnt[v]` là số đoạn có `end = v`. Tổng tiền tố `pre = list(accumulate(cnt, initial=0))` cho `pre[v] = cnt[0] + ... + cnt[v−1]` = số đoạn có `end < v`.

In [7]:
ends = [4, 5, 6, 4]
cnt = [0] * 8
for e in ends:
    cnt[e] += 1
pre = list(accumulate(cnt, initial=0))
print("cnt =", cnt)
print("pre =", pre)
print("số end < 5:", pre[5], "| kiểm tra:", sum(e < 5 for e in ends))    # 2

cnt = [0, 0, 0, 0, 2, 1, 1, 0]
pre = [0, 0, 0, 0, 0, 2, 3, 4, 4]
số end < 5: 2 | kiểm tra: 2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Vẽ ví dụ 1 trên trục số:

```
1 2 3 4
[-]           [1,2]
  [-]         [2,3]
    [-]       [3,4]
```

Xét từng cặp:

| cặp | hai đoạn | `r1 < l2` hoặc `r2 < l1`? | kết luận |
|---|---|---|---|
| (0,1) | `[1,2]`, `[2,3]` | `2 < 2` sai; `3 < 1` sai | giao nhau (tại 2) |
| (0,2) | `[1,2]`, `[3,4]` | `2 < 3` **đúng** | rời nhau |
| (1,2) | `[2,3]`, `[3,4]` | `3 < 3` sai; `4 < 2` sai | giao nhau (tại 3) |

Có `3` cặp; `1` cặp rời nhau; vậy `3 − 1 = 2` cặp giao nhau. Hai cách nghĩ đều đúng: **đếm trực tiếp từng cặp** hoặc **lấy tổng trừ số cặp rời nhau**.

In [8]:
intervals = [[1, 2], [2, 3], [3, 4]]
total = 0
for i, j in combinations(range(len(intervals)), 2):
    a, b = intervals[i], intervals[j]
    hit = max(a[0], b[0]) <= min(a[1], b[1])
    total += hit
    print(f"cặp ({i},{j}): {a} và {b} -> {'giao nhau' if hit else 'rời nhau'}")
print("số cặp giao nhau:", total)

cặp (0,1): [1, 2] và [2, 3] -> giao nhau
cặp (0,2): [1, 2] và [3, 4] -> rời nhau
cặp (1,2): [2, 3] và [3, 4] -> giao nhau
số cặp giao nhau: 2


## 5. Cách 1 — Brute Force / baseline: duyệt mọi cặp

**Ý tưởng.** Duyệt mọi cặp `i < j` bằng hai vòng `for` lồng nhau; với mỗi cặp kiểm tra `max(start_i, start_j) <= min(end_i, end_j)`; đếm những cặp thỏa.

In [9]:
def count_pairs_bruteforce(intervals: list[list[int]]) -> int:
    n = len(intervals)
    answer = 0
    for i in range(n):
        for j in range(i + 1, n):                    # i < j: mỗi cặp đúng một lần
            l1, r1 = intervals[i]
            l2, r2 = intervals[j]
            if max(l1, l2) <= min(r1, r2):           # có điểm chung (kể cả chạm đầu mút)
                answer += 1
    return answer


for intervals, expected in examples:
    print(f"intervals={intervals} -> {count_pairs_bruteforce(intervals)} (kỳ vọng {expected})")

intervals=[[1, 2], [2, 3], [3, 4]] -> 2 (kỳ vọng 2)
intervals=[[1, 5], [2, 4], [3, 6]] -> 3 (kỳ vọng 3)
intervals=[[1, 2], [3, 4], [5, 6]] -> 0 (kỳ vọng 0)


**Độ phức tạp (Cách 1).** Có `n(n−1)/2` cặp, mỗi cặp so sánh `O(1)`: **Time `O(n²)`**, **Space `O(1)`**. Với `n ≤ 100` cách này nộp được ngay (tối đa 4950 cặp).

**Nút thắt.** Ta kiểm tra từng cặp một dù nhiều cặp "chắc chắn rời nhau" hoặc "chắc chắn giao nhau". Nếu biết sắp xếp, ta đếm **hàng loạt** thay vì từng cặp. Mục tiêu của các cách sau: `O(n log n)` hoặc tốt hơn.

## 6. Ý tưởng thuật toán / bước đột phá

**Đếm phần bù.** Đếm cặp **rời nhau** rồi lấy `n(n−1)/2` trừ đi. Lợi ích: một cặp rời nhau luôn có một đoạn **kết thúc trước hẳn**, tức `end_A < start_B`. Chỉ cần đếm với mỗi đoạn `B`: "có bao nhiêu đoạn `A` có `end_A < start_B`?" — một câu hỏi chỉ phụ thuộc `start_B` và **tập các `end`**, không cần biết `A` ghép với `start` nào.

**Mỗi cặp rời nhau đếm đúng một lần.** Nếu `A` và `B` rời nhau với `end_A < start_B`, thì `end_B ≥ start_B > end_A ≥ start_A`, nên `end_B < start_A` **không thể** đúng. Vậy cặp `{A, B}` chỉ được đếm khi xét `B` (qua `A`), không bao giờ khi xét `A` (qua `B`). Cũng không có đoạn nào tự đếm chính nó vì `end ≥ start`.

Do đó:

```
disjoint = Σ_B #{ A : end_A < start_B }
answer   = n(n−1)/2 − disjoint
```

Việc đếm `#{A : end_A < s}` làm nhanh bằng **mảng `ends` đã sắp xếp** (`bisect_left`), bằng **hai con trỏ** (khi `starts` cũng sắp xếp), hoặc bằng **mảng đếm + tổng tiền tố**.

In [10]:
intervals = [[1, 2], [2, 3], [3, 4]]
n = len(intervals)
ends = sorted(e for s, e in intervals)
disjoint = 0
for s, e in intervals:
    before = bisect_left(ends, s)           # số đoạn kết thúc trước hẳn s
    print(f"đoạn [{s},{e}]: có {before} đoạn có end < {s}")
    disjoint += before
print("số cặp rời nhau:", disjoint, "-> đáp án:", n * (n - 1) // 2 - disjoint)

đoạn [1,2]: có 0 đoạn có end < 1
đoạn [2,3]: có 0 đoạn có end < 2
đoạn [3,4]: có 1 đoạn có end < 3
số cặp rời nhau: 1 -> đáp án: 2


## 7. Các cách tiếp theo

### Cách 2 — Phần bù với `ends` đã sắp xếp và `bisect_left`

**Giải thích.**

1. `ends = sorted(end của mọi đoạn)`.
2. Với mỗi đoạn có `start = s`: cộng `bisect_left(ends, s)` (số `end < s`) vào `disjoint`.
3. Đáp án `n*(n−1)//2 − disjoint`.

In [11]:
def count_pairs_bisect(intervals: list[list[int]]) -> int:
    n = len(intervals)
    ends = sorted(e for _, e in intervals)
    disjoint = 0
    for s, _ in intervals:
        disjoint += bisect_left(ends, s)          # số đoạn kết thúc TRƯỚC HẲN s (end < s)
    return n * (n - 1) // 2 - disjoint


for intervals, expected in examples:
    print(f"intervals={intervals} -> {count_pairs_bisect(intervals)} (kỳ vọng {expected})")

intervals=[[1, 2], [2, 3], [3, 4]] -> 2 (kỳ vọng 2)
intervals=[[1, 5], [2, 4], [3, 6]] -> 3 (kỳ vọng 3)
intervals=[[1, 2], [3, 4], [5, 6]] -> 0 (kỳ vọng 0)


In [12]:
# Code nộp LeetCode cho Cách 2
from bisect import bisect_left
from typing import List


class Solution:
    def countIntersectingIntervals(self, intervals: List[List[int]]) -> int:
        n = len(intervals)
        ends = sorted(e for _, e in intervals)
        disjoint = sum(bisect_left(ends, s) for s, _ in intervals)
        return n * (n - 1) // 2 - disjoint

**Độ phức tạp Cách 2:** **Time `O(n log n)`** (sắp xếp `n` phần tử, rồi `n` lần `bisect` mỗi lần `O(log n)`), **Space `O(n)`** cho `ends`.

### Cách 3 — Sắp xếp hai mảng riêng và hai con trỏ (cách của lời giải tham khảo)

**Giải thích.** Sắp `starts` và `ends` riêng, tăng dần. Duyệt từng `start` theo thứ tự tăng và giữ một con trỏ `i` sao cho mọi `ends[0..i-1] < start`. Khi `start` tăng, `i` chỉ tiến lên, không bao giờ lùi: tổng số bước của `i` là `n`. Mỗi lần ta trừ `i` vào đáp án.

In [13]:
def count_pairs_two_pointers(intervals: list[list[int]]) -> int:
    n = len(intervals)
    starts = sorted(s for s, _ in intervals)
    ends = sorted(e for _, e in intervals)
    answer = n * (n - 1) // 2
    i = 0                                        # số end đã biết là < start hiện tại
    for s in starts:
        while i < n and ends[i] < s:
            i += 1
        answer -= i                              # i đoạn kết thúc trước hẳn s: i cặp rời nhau
    return answer


for intervals, expected in examples:
    print(f"intervals={intervals} -> {count_pairs_two_pointers(intervals)} (kỳ vọng {expected})")

intervals=[[1, 2], [2, 3], [3, 4]] -> 2 (kỳ vọng 2)
intervals=[[1, 5], [2, 4], [3, 6]] -> 3 (kỳ vọng 3)
intervals=[[1, 2], [3, 4], [5, 6]] -> 0 (kỳ vọng 0)


**Độ phức tạp Cách 3:** **Time `O(n log n)`** (hai lần sắp xếp; hai con trỏ chỉ `O(n)`), **Space `O(n)`**.

### Cách 4 — Mảng đếm và tổng tiền tố (tận dụng giá trị nhỏ ≤ 100)

**Giải thích.** Vì `0 <= end <= 100`, dùng `cnt[v]` đếm số đoạn có `end = v`, rồi `pre = accumulate(cnt, initial=0)` để `pre[v]` là số đoạn có `end < v`. Với mỗi đoạn, `disjoint += pre[start]`. Không cần sắp xếp.

In [14]:
def count_pairs_counting(intervals: list[list[int]]) -> int:
    n = len(intervals)
    top = max(e for _, e in intervals)
    cnt = [0] * (top + 1)
    for _, e in intervals:
        cnt[e] += 1                               # cnt[v] = số đoạn có end = v
    pre = list(accumulate(cnt, initial=0))        # pre[v] = số đoạn có end < v
    disjoint = 0
    for s, _ in intervals:
        disjoint += pre[min(s, top + 1)]          # s <= end <= top nên s <= top, min chỉ để an toàn
    return n * (n - 1) // 2 - disjoint


for intervals, expected in examples:
    print(f"intervals={intervals} -> {count_pairs_counting(intervals)} (kỳ vọng {expected})")

intervals=[[1, 2], [2, 3], [3, 4]] -> 2 (kỳ vọng 2)
intervals=[[1, 5], [2, 4], [3, 6]] -> 3 (kỳ vọng 3)
intervals=[[1, 2], [3, 4], [5, 6]] -> 0 (kỳ vọng 0)


**Độ phức tạp Cách 4:** **Time `O(n + V)`** với `V` là giá trị lớn nhất (≤ 100): tuyến tính, không sắp xếp. **Space `O(V)`**.

### Cách 5 — Quét đường thẳng (sweep line) với các sự kiện

**Giải thích.** Biến mỗi đoạn thành hai sự kiện: **bắt đầu** tại `start` và **kết thúc** tại `end`. Sắp xếp các sự kiện theo tọa độ, và khi **cùng tọa độ thì sự kiện bắt đầu đứng trước sự kiện kết thúc** (vì đoạn đóng chạm đầu mút vẫn giao nhau). Duyệt: giữ `active` là số đoạn đang "mở". Khi gặp **bắt đầu**: đoạn mới giao với mọi đoạn đang mở, nên `answer += active`, rồi `active += 1`. Khi gặp **kết thúc**: `active -= 1`.

In [15]:
def count_pairs_sweep(intervals: list[list[int]]) -> int:
    events = []
    for s, e in intervals:
        events.append((s, 0))            # 0 = bắt đầu (đứng trước ở cùng tọa độ)
        events.append((e, 1))            # 1 = kết thúc
    events.sort()
    active = 0
    answer = 0
    for _, kind in events:
        if kind == 0:
            answer += active             # đoạn mới giao với mọi đoạn còn mở
            active += 1
        else:
            active -= 1
    return answer


for intervals, expected in examples:
    print(f"intervals={intervals} -> {count_pairs_sweep(intervals)} (kỳ vọng {expected})")

intervals=[[1, 2], [2, 3], [3, 4]] -> 2 (kỳ vọng 2)
intervals=[[1, 5], [2, 4], [3, 6]] -> 3 (kỳ vọng 3)
intervals=[[1, 2], [3, 4], [5, 6]] -> 0 (kỳ vọng 0)


**Độ phức tạp Cách 5:** **Time `O(n log n)`** (sắp xếp `2n` sự kiện), **Space `O(n)`**. Ưu điểm: không cần ý tưởng "đếm phần bù", và mở rộng được cho nhiều bài khoảng khác (độ sâu lớn nhất, v.v.).

> Cách 2 và Cách 3 cùng ý (đếm phần bù), Cách 4 nhanh nhất khi giá trị nhỏ, Cách 5 là kỹ thuật tổng quát. Cách 1 vẫn là chuẩn đối chiếu.

## 8. TRACE từng bước

Ta theo dõi Cách 3 (hai con trỏ) trên ví dụ 3 (`[[1,2],[3,4],[5,6]]`, đáp án `0`) và ví dụ 1, rồi Cách 5 (sweep) trên ví dụ 2.

In [16]:
def trace_two_pointers(intervals):
    n = len(intervals)
    starts = sorted(s for s, _ in intervals)
    ends = sorted(e for _, e in intervals)
    answer = n * (n - 1) // 2
    print(f"intervals={intervals}")
    print(f"starts={starts}, ends={ends}, tổng cặp = {answer}")
    i = 0
    for s in starts:
        print(f"  start={s}: i={i}")
        while i < n and ends[i] < s:
            print(f"    ends[{i}]={ends[i]} < {s} -> đoạn kết thúc trước, i = {i + 1}")
            i += 1
        if i < n:
            print(f"    ends[{i}]={ends[i]} >= {s} -> dừng")
        answer -= i
        print(f"    trừ {i} -> answer = {answer}")
    print("  => đáp án:", answer)
    return answer


assert trace_two_pointers([[1, 2], [3, 4], [5, 6]]) == 0
print()
assert trace_two_pointers([[1, 2], [2, 3], [3, 4]]) == 2

intervals=[[1, 2], [3, 4], [5, 6]]
starts=[1, 3, 5], ends=[2, 4, 6], tổng cặp = 3
  start=1: i=0
    ends[0]=2 >= 1 -> dừng
    trừ 0 -> answer = 3
  start=3: i=0
    ends[0]=2 < 3 -> đoạn kết thúc trước, i = 1
    ends[1]=4 >= 3 -> dừng
    trừ 1 -> answer = 2
  start=5: i=1
    ends[1]=4 < 5 -> đoạn kết thúc trước, i = 2
    ends[2]=6 >= 5 -> dừng
    trừ 2 -> answer = 0
  => đáp án: 0

intervals=[[1, 2], [2, 3], [3, 4]]
starts=[1, 2, 3], ends=[2, 3, 4], tổng cặp = 3
  start=1: i=0
    ends[0]=2 >= 1 -> dừng
    trừ 0 -> answer = 3
  start=2: i=0
    ends[0]=2 >= 2 -> dừng
    trừ 0 -> answer = 3
  start=3: i=0
    ends[0]=2 < 3 -> đoạn kết thúc trước, i = 1
    ends[1]=3 >= 3 -> dừng
    trừ 1 -> answer = 2
  => đáp án: 2


In [17]:
def trace_sweep(intervals):
    events = []
    for s, e in intervals:
        events.append((s, 0))
        events.append((e, 1))
    events.sort()
    print(f"intervals={intervals}")
    print("sự kiện đã sắp (tọa độ, loại 0=bắt đầu 1=kết thúc):", events)
    active = answer = 0
    for x, kind in events:
        if kind == 0:
            answer += active
            print(f"  x={x} BẮT ĐẦU: active={active} -> answer += {active} => {answer}; active = {active + 1}")
            active += 1
        else:
            print(f"  x={x} KẾT THÚC: active = {active} -> {active - 1}")
            active -= 1
    print("  => đáp án:", answer)
    return answer


assert trace_sweep([[1, 5], [2, 4], [3, 6]]) == 3

intervals=[[1, 5], [2, 4], [3, 6]]
sự kiện đã sắp (tọa độ, loại 0=bắt đầu 1=kết thúc): [(1, 0), (2, 0), (3, 0), (4, 1), (5, 1), (6, 1)]
  x=1 BẮT ĐẦU: active=0 -> answer += 0 => 0; active = 1
  x=2 BẮT ĐẦU: active=1 -> answer += 1 => 1; active = 2
  x=3 BẮT ĐẦU: active=2 -> answer += 2 => 3; active = 3
  x=4 KẾT THÚC: active = 3 -> 2
  x=5 KẾT THÚC: active = 2 -> 1
  x=6 KẾT THÚC: active = 1 -> 0
  => đáp án: 3


Bảng tóm tắt (Cách 3, ví dụ 1: `starts = [1,2,3]`, `ends = [2,3,4]`, tổng cặp `3`):

| `start` | các `end < start` | `i` sau khi tiến | trừ | `answer` |
|---|---|---|---|---|
| 1 | không có | 0 | 0 | 3 |
| 2 | không có (`ends[0] = 2` không `< 2`) | 0 | 0 | 3 |
| 3 | `2` (`ends[0]`) | 1 | 1 | 2 |

Điểm mấu chốt ở hàng `start = 2`: `ends[0] = 2` **bằng** `start`, không **nhỏ hơn**, nên đoạn kết thúc tại `2` chưa "kết thúc trước hẳn" mà chạm đầu mút, hai đoạn vẫn giao nhau. Kết quả `2` khớp đề.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (sắp xếp hai mảng riêng và hai con trỏ), đúng ý lời giải tham khảo. Đúng class `Solution`, đúng method `countIntersectingIntervals(self, intervals: List[List[int]]) -> int`.

In [18]:
from typing import List


class Solution:
    def countIntersectingIntervals(self, intervals: List[List[int]]) -> int:
        n = len(intervals)
        starts = sorted(s for s, _ in intervals)
        ends = sorted(e for _, e in intervals)
        answer = n * (n - 1) // 2                 # tổng số cặp i < j
        i = 0                                     # số đoạn đã kết thúc trước hẳn start hiện tại
        for s in starts:
            while i < n and ends[i] < s:          # chặt: end == start vẫn giao nhau
                i += 1
            answer -= i                           # trừ các cặp rời nhau
        return answer


for intervals, expected in examples:
    assert Solution().countIntersectingIntervals(intervals) == expected
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Giao nhau ⇔ không rời nhau:** hai đoạn đóng rời nhau khi và chỉ khi `r1 < l2` hoặc `r2 < l1`. Điều kiện phủ định là `l2 <= r1` và `l1 <= r2`, tức `max(l) <= min(r)`: có điểm chung.
- **Phân loại cặp rời nhau theo "ai kết thúc trước":** với mọi cặp rời nhau `{A, B}`, đúng một trong hai bất đẳng thức `r_A < l_B`, `r_B < l_A` xảy ra (không thể cùng lúc vì `l ≤ r`: nếu cả hai đều đúng thì `r_A < l_B ≤ r_B < l_A ≤ r_A`, vô lý).
- **Đếm đúng một lần:** tổng `Σ_B #{A : r_A < l_B}` đếm mỗi cặp rời nhau đúng một lần (qua đoạn bắt đầu muộn hơn). Một đoạn không tự đếm vì `r_B ≥ l_B` nên `r_B < l_B` sai.
- **Tách khỏi quan hệ ghép:** `#{A : r_A < l_B}` chỉ phụ thuộc vào **tập** các `r_A`, nên ta có thể sắp `starts` và `ends` **độc lập** mà không mất thông tin cần thiết.
- **Hai con trỏ:** vì `starts` tăng dần, tập `{A : r_A < s}` chỉ nở ra khi `s` tăng; con trỏ `i` chỉ tiến lên.
- **Sweep line (Cách 5):** ở cùng tọa độ, sự kiện bắt đầu đứng trước kết thúc nên các đoạn chạm đầu mút vẫn cùng "mở" tại một thời điểm; mỗi cặp giao nhau được đếm đúng một lần tại thời điểm đoạn bắt đầu muộn hơn bắt đầu.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số đoạn, `V` là giá trị tối đa của toạ độ (≤ 100).

- **Cách 1 (mọi cặp):** Time `O(n²)`, Space `O(1)`.
- **Cách 2 (`bisect` trên `ends`):** Time `O(n log n)`, Space `O(n)`.
- **Cách 3 (hai con trỏ):** Time `O(n log n)` (sắp xếp), Space `O(n)`.
- **Cách 4 (mảng đếm + tổng tiền tố):** Time `O(n + V)`, Space `O(V)`.
- **Cách 5 (sweep line):** Time `O(n log n)`, Space `O(n)`.
- Trường hợp trung bình và xấu nhất như nhau cho mọi cách (đều không phụ thuộc cấu trúc dữ liệu băm hay ngẫu nhiên).

## 12. Bộ test

Ta gom các cách vào từ điển `impls` và viết một **lời giải độc lập** `oracle`: với mỗi cặp, so sánh **tập các điểm nguyên** `set(range(l, r + 1))` của hai đoạn (toạ độ nguyên nên hai đoạn đóng có điểm chung ⇔ có điểm nguyên chung).

In [19]:
sol = Solution()
impls = {
    "bruteforce": count_pairs_bruteforce,
    "bisect": count_pairs_bisect,
    "two_pointers": count_pairs_two_pointers,
    "counting": count_pairs_counting,
    "sweep": count_pairs_sweep,
    "submit": sol.countIntersectingIntervals,
}


def oracle(intervals):
    """Đáp án độc lập: so sánh tập điểm nguyên của từng cặp đoạn."""
    covered = [set(range(l, r + 1)) for l, r in intervals]
    n = len(intervals)
    return sum(1 for i in range(n) for j in range(i + 1, n) if covered[i] & covered[j])


for intervals, expected in examples:
    assert oracle(intervals) == expected
    for name, fn in impls.items():
        assert fn([list(x) for x in intervals]) == expected, (name, intervals, fn(intervals), expected)
print("PASS ví dụ của đề (mọi cách và oracle độc lập)")

PASS ví dụ của đề (mọi cách và oracle độc lập)


### 12.1. Các trường hợp biên

In [20]:
def check_all(intervals, expected, label):
    assert oracle(intervals) == expected, (label, "oracle", oracle(intervals), expected)
    for name, fn in impls.items():
        got = fn([list(x) for x in intervals])
        assert got == expected, (label, name, got, expected)
    print("PASS", label, "->", expected)


check_all([[1, 2], [1, 2]], 1, "n = 2, hai đoạn giống hệt nhau")
check_all([[1, 2], [3, 4]], 0, "n = 2, rời nhau")
check_all([[1, 2], [2, 4]], 1, "n = 2, chạm đúng một đầu mút")
check_all([[0, 0], [0, 0], [0, 0]], 3, "ba đoạn điểm cùng tọa độ 0")
check_all([[3, 3], [3, 3], [4, 4]], 1, "đoạn điểm: hai cái trùng nhau, một cái khác")
check_all([[0, 100], [50, 50], [100, 100], [0, 0]], 3, "đoạn bao trùm và đoạn điểm ở biên")
check_all([[3, 4], [1, 5]], 1, "đoạn đứng sau bắt đầu sớm hơn (lồng nhau)")
check_all([[1, 10], [2, 3], [4, 5], [6, 7]], 3, "một đoạn lớn chứa ba đoạn nhỏ rời nhau")
check_all([[0, 100]] * 100, 4950, "n = 100, mọi đoạn giống nhau: C(100, 2)")
check_all([[i, i] for i in range(100)], 0, "n = 100, các đoạn điểm đều khác nhau")
check_all([[i, i + 1] for i in range(100)], 99, "n = 100, mỗi đoạn chạm đoạn kế tiếp")

PASS n = 2, hai đoạn giống hệt nhau -> 1
PASS n = 2, rời nhau -> 0
PASS n = 2, chạm đúng một đầu mút -> 1
PASS ba đoạn điểm cùng tọa độ 0 -> 3
PASS đoạn điểm: hai cái trùng nhau, một cái khác -> 1
PASS đoạn bao trùm và đoạn điểm ở biên -> 3
PASS đoạn đứng sau bắt đầu sớm hơn (lồng nhau) -> 1
PASS một đoạn lớn chứa ba đoạn nhỏ rời nhau -> 3
PASS n = 100, mọi đoạn giống nhau: C(100, 2) -> 4950
PASS n = 100, các đoạn điểm đều khác nhau -> 0
PASS n = 100, mỗi đoạn chạm đoạn kế tiếp -> 99


### 12.2. Tính chất và phản ví dụ

In [21]:
random.seed(4056)


def random_intervals(n, hi=100):
    out = []
    for _ in range(n):
        a, b = random.randint(0, hi), random.randint(0, hi)
        out.append([min(a, b), max(a, b)])
    return out


# Tính chất 1: tịnh tiến mọi tọa độ không đổi đáp án (miễn còn trong 0..100)
for _ in range(300):
    n = random.randint(2, 12)
    iv = random_intervals(n, 50)
    shifted = [[l + 50, r + 50] for l, r in iv]
    assert sol.countIntersectingIntervals(iv) == sol.countIntersectingIntervals(shifted)
print("PASS tính chất: tịnh tiến không đổi đáp án")

# Tính chất 2: đáp án không phụ thuộc thứ tự các đoạn; nằm trong [0, n(n-1)/2]
for _ in range(300):
    n = random.randint(2, 15)
    iv = random_intervals(n)
    shuffled = iv[:]
    random.shuffle(shuffled)
    a = sol.countIntersectingIntervals(iv)
    assert a == sol.countIntersectingIntervals(shuffled)
    assert 0 <= a <= n * (n - 1) // 2
print("PASS tính chất: hoán vị các đoạn không đổi đáp án; 0 <= đáp án <= C(n,2)")

# Tính chất 3: giao nhau + rời nhau = C(n, 2) (tính rời nhau bằng quan hệ riêng)
for _ in range(300):
    n = random.randint(2, 15)
    iv = random_intervals(n)
    disj = sum(1 for (a, b), (c, d) in combinations(iv, 2) if b < c or d < a)
    assert sol.countIntersectingIntervals(iv) + disj == n * (n - 1) // 2
print("PASS tính chất: giao nhau + rời nhau = C(n,2)")

# Phản ví dụ: dùng '<' chặt làm mất các cặp chạm đầu mút
def sai_chat(iv):
    n = len(iv)
    return sum(1 for i in range(n) for j in range(i + 1, n)
               if max(iv[i][0], iv[j][0]) < min(iv[i][1], iv[j][1]))

print("[[1,2],[2,3],[3,4]] -> sai:", sai_chat([[1, 2], [2, 3], [3, 4]]), "| đúng:", sol.countIntersectingIntervals([[1, 2], [2, 3], [3, 4]]))
assert sai_chat([[1, 2], [2, 3], [3, 4]]) == 0

PASS tính chất: tịnh tiến không đổi đáp án
PASS tính chất: hoán vị các đoạn không đổi đáp án; 0 <= đáp án <= C(n,2)
PASS tính chất: giao nhau + rời nhau = C(n,2)
[[1,2],[2,3],[3,4]] -> sai: 0 | đúng: 2


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [22]:
random.seed(2026)
for trial in range(2500):
    n = random.randint(2, 12)
    hi = random.choice([3, 6, 10, 30, 100])
    iv = random_intervals(n, hi)
    expected = oracle(iv)
    for name, fn in impls.items():
        got = fn([list(x) for x in iv])
        assert got == expected, (name, iv, got, expected)

for trial in range(150):                       # n tới 100
    n = random.randint(50, 100)
    hi = random.choice([10, 40, 100])
    iv = random_intervals(n, hi)
    expected = oracle(iv)
    for name, fn in impls.items():
        got = fn([list(x) for x in iv])
        assert got == expected, (name, got, expected)
print("PASS 2650 input ngẫu nhiên: mọi cách khớp oracle độc lập")

PASS 2650 input ngẫu nhiên: mọi cách khớp oracle độc lập


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `<` chặt thay vì `<=` khi kiểm tra giao nhau

In [23]:
def sai_chat_hon(intervals):
    n = len(intervals)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            l1, r1 = intervals[i]
            l2, r2 = intervals[j]
            if max(l1, l2) < min(r1, r2):          # sai: đoạn chạm đầu mút bị loại
                count += 1
    return count


print("sai:", sai_chat_hon([[1, 2], [2, 3], [3, 4]]), "| đúng:", Solution().countIntersectingIntervals([[1, 2], [2, 3], [3, 4]]))

sai: 0 | đúng: 2


**Vì sao sai:** đề nói rõ hai đoạn **chung một đầu mút vẫn giao nhau**. Đoạn đóng `[1,2]` và `[2,3]` cùng chứa điểm `2`; điều kiện đúng là `max(l) <= min(r)` (dấu `<=`). Với `<` ta chỉ đếm các cặp có phần giao có độ dài dương.

### Lỗi 2: Đếm cặp có thứ tự (gấp đôi)

In [24]:
def sai_cap_co_thu_tu(intervals):
    n = len(intervals)
    count = 0
    for i in range(n):
        for j in range(n):                          # sai: j chạy từ 0, và có cả i == j
            l1, r1 = intervals[i]
            l2, r2 = intervals[j]
            if i != j and max(l1, l2) <= min(r1, r2):
                count += 1
    return count


print("sai:", sai_cap_co_thu_tu([[1, 5], [2, 4], [3, 6]]), "| đúng:", Solution().countIntersectingIntervals([[1, 5], [2, 4], [3, 6]]))

sai: 6 | đúng: 3


**Vì sao sai:** đề đếm cặp **`i < j`**, mỗi cặp một lần. Duyệt cả `(i, j)` và `(j, i)` làm kết quả gấp đôi (`6` thay vì `3`). Dùng `range(i + 1, n)` hoặc chia kết quả cho 2.

### Lỗi 3: Dùng `<=` ở con trỏ hai chiều

In [25]:
def sai_con_tro_le(intervals):
    n = len(intervals)
    starts = sorted(s for s, _ in intervals)
    ends = sorted(e for _, e in intervals)
    answer = n * (n - 1) // 2
    i = 0
    for s in starts:
        while i < n and ends[i] <= s:              # sai: end == start vẫn giao nhau
            i += 1
        answer -= i
    return answer


print("sai:", sai_con_tro_le([[1, 2], [2, 3], [3, 4]]), "| đúng:", Solution().countIntersectingIntervals([[1, 2], [2, 3], [3, 4]]))

sai: 0 | đúng: 2


**Vì sao sai:** `ends[i] <= s` coi đoạn kết thúc tại đúng `s` là "rời nhau", trong khi nó chạm đầu mút và giao nhau. Điều kiện đúng cho phần bù là **`ends[i] < s`** (chặt).

### Lỗi 4: Chỉ kiểm tra một chiều, bỏ sót đoạn bao trùm

In [26]:
def sai_mot_chieu(intervals):
    n = len(intervals)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            # chỉ kiểm tra start của đoạn j có nằm trong đoạn i không
            if intervals[i][0] <= intervals[j][0] <= intervals[i][1]:
                count += 1
    return count


print("sai:", sai_mot_chieu([[3, 4], [1, 5]]), "| đúng:", Solution().countIntersectingIntervals([[3, 4], [1, 5]]))

sai: 0 | đúng: 1


**Vì sao sai:** hai đoạn giao nhau cũng khi `start` của đoạn `i` nằm trong đoạn `j` (đoạn đứng sau bắt đầu sớm hơn). `[3,4]` và `[1,5]` giao nhau nhưng `1` không nằm trong `[3,4]`. Phải kiểm tra **cả hai chiều** (`l1 <= r2` và `l2 <= r1`, tức `max(l) <= min(r)`).

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**giao nhau / chồng lấn / overlap** của các đoạn": nghĩ ngay đến **`max(l) <= min(r)`** và **sắp xếp**.
- Muốn đếm "có bao nhiêu cặp thỏa X" mà X khó, thử **đếm phần bù** (`tổng − số cặp không thỏa`), đặc biệt khi phần bù có cấu trúc một chiều (`end_A < start_B`).
- Đoạn **đóng** (chạm đầu mút vẫn giao) → dùng `<` chặt cho "rời nhau"; đoạn **nửa mở** `[l, r)` thì chạm đầu mút không giao (dùng `<=` cho "rời nhau").
- Khi `start` và `end` được sắp xếp **riêng**, ta mất sự ghép đôi, nhưng các bài chỉ cần "số `end` nhỏ hơn `s`" thì không cần ghép đôi.
- Giá trị nhỏ (≤ 100) gợi ý **mảng đếm + tổng tiền tố** thay cho sắp xếp.
- Kỹ thuật **sweep line** (sắp sự kiện, giữ `active`) là khung chung cho các bài lịch họp, độ phủ lớn nhất, ghép đoạn.

## 15. Biến thể của bài

1. **Giao nhau theo độ dài dương** (chạm đầu mút không tính): đổi `<=` thành `<`.
2. **Độ sâu lớn nhất:** số đoạn chồng nhau nhiều nhất tại một điểm (LeetCode 253, Meeting Rooms II).
3. **Danh sách các cặp** giao nhau thay vì chỉ đếm (đối chiếu `combinations`).
4. **Số cặp lồng nhau** (một đoạn nằm trọn trong đoạn kia).
5. **Kiểm tra có hai đoạn nào giao nhau không** (LeetCode 252, Meeting Rooms): sắp theo `start`, so sánh các đoạn liền kề.

In [27]:
def count_pairs_strict(intervals):
    """Biến thể 1: phần giao có độ dài dương (đoạn nửa mở)."""
    n = len(intervals)
    return sum(1 for i in range(n) for j in range(i + 1, n)
               if max(intervals[i][0], intervals[j][0]) < min(intervals[i][1], intervals[j][1]))


def max_overlap(intervals):
    """Biến thể 2: số đoạn chồng nhau nhiều nhất tại một điểm (đoạn đóng)."""
    events = []
    for s, e in intervals:
        events.append((s, 0))
        events.append((e, 1))
    events.sort()
    active = best = 0
    for _, kind in events:
        if kind == 0:
            active += 1
            best = max(best, active)
        else:
            active -= 1
    return best


def intersecting_pairs_list(intervals):
    """Biến thể 3: các cặp chỉ số (i, j) giao nhau."""
    n = len(intervals)
    return [(i, j) for i in range(n) for j in range(i + 1, n)
            if max(intervals[i][0], intervals[j][0]) <= min(intervals[i][1], intervals[j][1])]


def nested_pairs(intervals):
    """Biến thể 4: số cặp lồng nhau (đoạn này chứa đoạn kia)."""
    n = len(intervals)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            (a, b), (c, d) = intervals[i], intervals[j]
            if (a <= c and d <= b) or (c <= a and b <= d):
                count += 1
    return count


assert count_pairs_strict([[1, 2], [2, 3], [3, 4]]) == 0
assert count_pairs_strict([[1, 5], [2, 4], [3, 6]]) == 3
assert max_overlap([[1, 5], [2, 4], [3, 6]]) == 3
assert max_overlap([[1, 2], [2, 3], [3, 4]]) == 2
assert intersecting_pairs_list([[1, 2], [2, 3], [3, 4]]) == [(0, 1), (1, 2)]
assert nested_pairs([[1, 10], [2, 3], [4, 5], [6, 7]]) == 3
print("PASS biến thể")

PASS biến thể


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp | `max(l) <= min(r)` cho từng cặp | `O(n²)` | `O(1)` | hiểu đề; đối chiếu; `n ≤ 100` |
| 2. Phần bù + `bisect` | `tổng − Σ bisect_left(ends, s)` | `O(n log n)` | `O(n)` | ngắn gọn, dễ nhớ |
| 3. Hai mảng sắp riêng + hai con trỏ | con trỏ `i` tiến theo `starts` tăng | `O(n log n)` | `O(n)` | **cách nộp bài** (khớp lời giải tham khảo) |
| 4. Mảng đếm + tổng tiền tố | `pre[s]` = số `end < s` | `O(n + V)` | `O(V)` | giá trị nhỏ, không sắp xếp |
| 5. Sweep line | sự kiện bắt đầu/kết thúc, `active` | `O(n log n)` | `O(n)` | kỹ thuật tổng quát cho bài đoạn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `max(1, 3) <= min(2, 4)`, `bisect_left([2, 3, 3, 5], 3)`, `5 * 4 // 2`, `Solution().countIntersectingIntervals([[0, 0], [0, 0]])`, `Solution().countIntersectingIntervals([[1, 2], [4, 5]])`.

**Bài 2 (tính tay).** Với `intervals = [[1,4],[2,3],[5,6],[4,5]]`: liệt kê 6 cặp, cho biết cặp nào giao nhau, rồi tính số cặp bằng cách đếm phần bù (`starts`, `ends` đã sắp).

**Bài 3 (điền TODO).** Hoàn thành `pairs_todo(intervals)` theo Cách 2 (`bisect_left` trên `ends`).

In [28]:
def pairs_todo(intervals):
    n = len(intervals)
    # TODO 1: ends = list các end đã sắp tăng
    # TODO 2: disjoint = tổng bisect_left(ends, s) cho mỗi start s
    # TODO 3: trả về n * (n - 1) // 2 - disjoint
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `pairs_sweep2(intervals)` theo Cách 5 (sweep line) mà không nhìn code mẫu. Kiểm tra bằng ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `can_attend_all(intervals)` trả `True` nếu **không có** hai đoạn nào giao nhau (đoạn đóng). (b) Viết `count_cross_pairs(a, b)` đếm số cặp `(x, y)` với `x` thuộc danh sách đoạn `a`, `y` thuộc danh sách đoạn `b`, giao nhau.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: đoạn `[1,2]` và `[3,4]` rời nhau; `bisect_left` đếm số phần tử nhỏ hơn `x`; `n(n-1)/2` với `n = 5`. Bài 2: vẽ các đoạn lên trục số.
- **Mức 2 (vừa):** Bài 1: `False` (`3 <= 2` sai), `1`, `10`, `1` (hai đoạn điểm trùng nhau), `0`. Bài 2: `starts = [1,2,4,5]`, `ends = [3,4,5,6]`: các cặp giao nhau là `(0,1)`, `(0,3)` (`[1,4]` và `[4,5]` chạm tại 4), `(2,3)` (`[5,6]` và `[4,5]` chạm tại 5); `(0,2)`, `(1,2)`, `(1,3)` rời nhau. Bài 3: ba TODO trong comment.
- **Mức 3 (sâu):** Bài 2: tính phần bù: `start 1: 0`, `start 2: 0`, `start 4: ends<4 là {3} -> 1`, `start 5: ends<5 là {3,4} -> 2` (chú ý `end = 5` không `< 5`): tổng 3; `6 − 3 = 3`. Bài 5(a): `max_overlap(...) <= 1`, hoặc dùng `Solution().countIntersectingIntervals(...) == 0`. Bài 5(b): cùng khung phần bù hoặc duyệt cặp `(x, y)` với `max(l) <= min(r)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert (max(1, 3) <= min(2, 4)) is False
assert bisect_left([2, 3, 3, 5], 3) == 1
assert 5 * 4 // 2 == 10
assert Solution().countIntersectingIntervals([[0, 0], [0, 0]]) == 1
assert Solution().countIntersectingIntervals([[1, 2], [4, 5]]) == 0

# Bài 2
iv2 = [[1, 4], [2, 3], [5, 6], [4, 5]]
assert intersecting_pairs_list(iv2) == [(0, 1), (0, 3), (2, 3)]
assert sorted(s for s, _ in iv2) == [1, 2, 4, 5] and sorted(e for _, e in iv2) == [3, 4, 5, 6]
assert Solution().countIntersectingIntervals(iv2) == 3

# Bài 3
def pairs_todo_ans(intervals):
    n = len(intervals)
    ends = sorted(e for _, e in intervals)
    disjoint = sum(bisect_left(ends, s) for s, _ in intervals)
    return n * (n - 1) // 2 - disjoint

for intervals, expected in examples:
    assert pairs_todo_ans(intervals) == expected

# Bài 4
def pairs_sweep2(intervals):
    events = sorted([(s, 0) for s, _ in intervals] + [(e, 1) for _, e in intervals])
    active = answer = 0
    for _, kind in events:
        if kind == 0:
            answer += active
            active += 1
        else:
            active -= 1
    return answer

for intervals, expected in examples:
    assert pairs_sweep2(intervals) == expected

# Bài 5(a)
def can_attend_all(intervals):
    return Solution().countIntersectingIntervals(intervals) == 0

assert can_attend_all([[1, 2], [3, 4], [5, 6]]) is True
assert can_attend_all([[1, 2], [2, 3]]) is False

# Bài 5(b)
def count_cross_pairs(a, b):
    return sum(1 for l1, r1 in a for l2, r2 in b if max(l1, l2) <= min(r1, r2))

assert count_cross_pairs([[1, 3], [5, 6]], [[3, 5], [7, 8]]) == 2
assert count_cross_pairs([[1, 2]], [[3, 4]]) == 0
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm cặp đoạn giao nhau bằng phần bù)

```python
class Solution:
    def countIntersectingIntervals(self, intervals: List[List[int]]) -> int:
        n = len(intervals)
        starts = sorted(s for s, _ in intervals)
        ends = sorted(e for _, e in intervals)
        answer = n * (n - 1) // 2
        i = 0
        for s in starts:
            while i < n and ends[i] < s:
                i += 1
            answer -= i
        return answer
```

**Checklist**

- [ ] Hai đoạn **đóng** giao nhau ⇔ `max(l) <= min(r)`; **rời nhau** ⇔ `r1 < l2` hoặc `r2 < l1` (dấu `<` chặt)?
- [ ] Đếm **cặp `i < j`** (mỗi cặp một lần), tổng là `n(n−1)/2`?
- [ ] Phần bù đếm đúng một lần cho mỗi cặp rời nhau (qua đoạn bắt đầu muộn hơn)?
- [ ] `starts` và `ends` được sắp **riêng**, con trỏ chỉ tiến lên với điều kiện `ends[i] < s`?
- [ ] Đã thử: chạm đầu mút (`[1,2]`, `[2,3]`), đoạn điểm, đoạn bao trùm, hai đoạn giống hệt, `n = 2`?
- [ ] Đã đối chiếu với brute force mọi cặp trên input ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def countIntersectingIntervals(self, intervals: List[List[int]]) -> int`, có `from typing import List`?